<a href="https://colab.research.google.com/github/donna6355/study_python/blob/master/dl/CLOVA_OCR_%E1%84%89%E1%85%A9%E1%86%AB%E1%84%80%E1%85%B3%E1%86%AF%E1%84%8A%E1%85%B5_%E1%84%89%E1%85%B5%E1%86%AF%E1%84%89%E1%85%B3%E1%86%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🇰🇷 한글 손글씨 OCR 실습 — 실제 필기체 인식

**실제 한글 손글씨 이미지**를 활용하여 OCR 인식 성능을 비교합니다.  
EasyOCR · Tesseract + (선택) CLOVA OCR

---

## 테스트 이미지: 실제 손글씨 (시 한 편)

> 낮은 곳에 있고 싶었다  
> 낮은 곳이라면 지상의  
> 그 어디라도 좋다  
> 찰랑찰랑 고여들 네 사랑을  
> 온몸으로 받아들일 수만 있다면  
> 한 방울도 헛되이  
> 새어 나가지 않게 할 수 있다면  

## 실습 구성

| Part | 내용 |
|------|------|
| **0** | 패키지 설치 + 이미지 업로드 |
| **1** | 이미지 전처리 (대비·샤프닝·이진화) |
| **2** | EasyOCR 인식 + 바운딩 박스 |
| **3** | Tesseract 인식 |
| **4** | 3종 비교 분석 + 시각화 |
| **5** | (선택) CLOVA OCR API |

In [1]:
# ══════════════════════════════════════════════
# 0-1. 패키지 설치
# ══════════════════════════════════════════════
!pip install requests pillow easyocr -qqq 2>/dev/null
!apt-get -qq install -y tesseract-ocr tesseract-ocr-kor fonts-nanum > /dev/null 2>&1
!pip install pytesseract -qqq 2>/dev/null

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
fe = fm.FontEntry(fname='/usr/share/fonts/truetype/nanum/NanumGothic.ttf', name='NanumGothic')
fm.fontManager.ttflist.insert(0, fe)
plt.rcParams['font.family'] = fe.name
plt.rcParams['axes.unicode_minus'] = False

print("✅ 설치 완료!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 12.5 MB/s eta 0:00:00
✅ 설치 완료!


In [ ]:
# ══════════════════════════════════════════════
# 0-2. 손글씨 이미지 업로드 + 정답 텍스트
# ══════════════════════════════════════════════
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance
import numpy as np
import os, time

IMG_PATH = 'handwriting_sample.png'

if not os.path.exists(IMG_PATH):
    from google.colab import files
    print("📤 손글씨 이미지를 업로드하세요:")
    uploaded = files.upload()
    IMG_PATH = list(uploaded.keys())[0]
    print(f"✅ 업로드 완료: {IMG_PATH}")
else:
    print(f"✅ 이미지 파일: {IMG_PATH}")

# 정답 텍스트
GROUND_TRUTH = """낮은 곳에 있고 싶었다
낮은 곳이라면 지상의
그 어디라도 좋다
찰랑찰랑 고여들 네 사랑을
온몸으로 받아들일 수만 있다면
한 방울도 헛되이
새어 나가지 않게 할 수 있다면"""

GT_LINES = [l.strip() for l in GROUND_TRUTH.strip().split('\n')]

# 원본 미리보기
img = Image.open(IMG_PATH)
fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(np.array(img))
ax.set_title(f'실제 한글 손글씨 ({img.size[0]}x{img.size[1]})', fontweight='bold', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.show()

print(f"\n📝 정답 텍스트 ({len(GT_LINES)}줄):")
for i, line in enumerate(GT_LINES):
    print(f"   {i+1}. {line}")

---
# Part 1. 이미지 전처리

OCR 성능을 높이기 위한 전처리 기법:
- **그레이스케일**: 색상 정보 제거
- **대비 향상**: 글자와 배경 차이 극대화
- **샤프닝**: 글자 경계 선명하게
- **이진화**: 흑백 2값으로 변환

In [ ]:
# ══════════════════════════════════════════════
# 1-1. 전처리 (그레이+대비+샤프닝+이진화)
# ══════════════════════════════════════════════

def preprocess_for_ocr(img_path, output_path='preprocessed.png'):
    img = Image.open(img_path)
    gray = img.convert('L')
    enhanced = ImageEnhance.Contrast(gray).enhance(1.5)
    sharpened = ImageEnhance.Sharpness(enhanced).enhance(2.0)
    arr = np.array(sharpened)
    threshold = np.mean(arr) - 20
    binary = ((arr < threshold) * 0 + (arr >= threshold) * 255).astype(np.uint8)
    Image.fromarray(binary).save(output_path)
    return output_path

preprocessed = preprocess_for_ocr(IMG_PATH)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(np.array(Image.open(IMG_PATH)))
axes[0].set_title('원본', fontweight='bold', fontsize=13)
axes[1].imshow(np.array(Image.open(preprocessed)), cmap='gray')
axes[1].set_title('전처리 (대비+샤프닝+이진화)', fontweight='bold', fontsize=13)
for ax in axes: ax.axis('off')
plt.suptitle('OCR 전처리 비교', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

---
# Part 2. EasyOCR 인식 + 바운딩 박스

In [ ]:
# ══════════════════════════════════════════════
# 2-1. EasyOCR 인식 (원본 + 전처리 비교)
# ══════════════════════════════════════════════
import easyocr
from difflib import SequenceMatcher
import re

def calc_sim(gt, pred):
    return SequenceMatcher(None, re.sub(r'\s+','',gt), re.sub(r'\s+','',pred)).ratio()

print("📥 EasyOCR 로드 중... (첫 실행 시 모델 다운로드)")
reader = easyocr.Reader(['ko', 'en'], gpu=True)
print("✅ 로드 완료!\n")

print("═" * 55)
print("  🔍 EasyOCR 한글 손글씨 인식")
print("═" * 55)

easy_best = None
for label, fpath in [('원본', IMG_PATH), ('전처리', preprocessed)]:
    start = time.time()
    result = reader.readtext(fpath)
    elapsed = time.time() - start

    texts = [r[1] for r in result]
    confs = [r[2] for r in result]
    full_text = '\n'.join(texts)
    sim = calc_sim(GROUND_TRUTH, full_text)

    if easy_best is None or sim > easy_best['sim']:
        easy_best = {'result': result, 'text': full_text, 'sim': sim,
                     'time': elapsed, 'label': label, 'file': fpath}

    emoji = '🟢' if sim > 0.7 else '🟡' if sim > 0.4 else '🔴'
    print(f"\n  {emoji} [{label}] 정확도: {sim:.1%} ({elapsed:.1f}초, {len(texts)}줄)")
    for t, c in zip(texts, confs):
        conf_emoji = '🟢' if c > 0.8 else '🟡' if c > 0.5 else '🔴'
        print(f"     {conf_emoji} [{c:.3f}] {t}")

print(f"\n  → 최고: [{easy_best['label']}] {easy_best['sim']:.1%}")

In [ ]:
# ══════════════════════════════════════════════
# 2-2. EasyOCR 바운딩 박스 시각화
# ══════════════════════════════════════════════

FONT = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

fig, ax = plt.subplots(figsize=(10, 8))
img_vis = Image.open(IMG_PATH).copy()
draw = ImageDraw.Draw(img_vis)

for bbox, text, conf in easy_best['result']:
    pts = [(int(p[0]), int(p[1])) for p in bbox]
    color = '#10B981' if conf > 0.8 else '#F97316' if conf > 0.5 else '#E94560'
    draw.polygon(pts, outline=color)
    draw.polygon([(p[0]+1,p[1]+1) for p in pts], outline=color)  # 두껍게
    try:
        sf = ImageFont.truetype(FONT, 13)
        draw.text((pts[0][0], max(0, pts[0][1]-16)),
                  f'{text[:10]} ({conf:.0%})', fill=color, font=sf)
    except: pass

ax.imshow(np.array(img_vis))
ax.set_title(f'EasyOCR 바운딩 박스 (정확도: {easy_best["sim"]:.1%})', fontweight='bold', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()

# 줄별 비교
print("\n  📋 줄별 비교 (정답 vs EasyOCR):")
easy_lines = [l for l in easy_best['text'].split('\n') if l.strip()]
max_n = max(len(GT_LINES), len(easy_lines))
print(f"  {'정답':<28} {'EasyOCR':<28} {'일치'}")
print(f"  {'─'*64}")
for i in range(max_n):
    gt = GT_LINES[i] if i < len(GT_LINES) else ''
    el = easy_lines[i] if i < len(easy_lines) else ''
    match = '✅' if calc_sim(gt, el) > 0.8 else '❌'
    print(f"  {gt:<28} {el:<28} {match}")

---
# Part 3. Tesseract 인식

In [ ]:
# ══════════════════════════════════════════════
# 3-1. Tesseract OCR (원본 + 전처리)
# ══════════════════════════════════════════════
import pytesseract

print("═" * 55)
print("  📝 Tesseract OCR")
print("═" * 55)

tess_best = None
for label, fpath in [('원본', IMG_PATH), ('전처리', preprocessed)]:
    start = time.time()
    text = pytesseract.image_to_string(
        Image.open(fpath), lang='kor', config='--psm 6'
    ).strip()
    elapsed = time.time() - start
    sim = calc_sim(GROUND_TRUTH, text)

    if tess_best is None or sim > tess_best['sim']:
        tess_best = {'text': text, 'sim': sim, 'time': elapsed, 'label': label}

    emoji = '🟢' if sim > 0.7 else '🟡' if sim > 0.4 else '🔴'
    print(f"\n  {emoji} [{label}] 정확도: {sim:.1%} ({elapsed:.1f}초)")
    for line in text.split('\n'):
        if line.strip():
            print(f"     {line.strip()}")

print(f"\n  → 최고: [{tess_best['label']}] {tess_best['sim']:.1%}")

---
# Part 4. 종합 비교 + 시각화

In [ ]:
# ══════════════════════════════════════════════
# 4-1. EasyOCR vs Tesseract 종합 비교
# ══════════════════════════════════════════════

results = {
    'EasyOCR':   {'text': easy_best['text'],  'sim': easy_best['sim'],  'time': easy_best['time']},
    'Tesseract': {'text': tess_best['text'],  'sim': tess_best['sim'],  'time': tess_best['time']},
}

print("═" * 60)
print("  📊 한글 손글씨 OCR 비교 (실제 필기체)")
print("═" * 60)
print(f"\n  {'엔진':<12} {'정확도':<10} {'시간(초)':<10} {'줄수':<6}")
print(f"  {'─'*38}")
for name, r in results.items():
    n_lines = len([l for l in r['text'].split('\n') if l.strip()])
    emoji = '🟢' if r['sim']>0.7 else '🟡' if r['sim']>0.4 else '🔴'
    print(f"  {emoji} {name:<10} {r['sim']:<10.1%} {r['time']:<10.1f} {n_lines}줄")

# 시각화
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
names = list(results.keys())
colors = ['#3B82F6', '#10B981']

sims = [results[n]['sim']*100 for n in names]
ax1.bar(names, sims, color=colors, width=0.5)
ax1.set_ylim(0, 105)
ax1.set_title('정확도 (%)', fontweight='bold', fontsize=13)
for i, v in enumerate(sims):
    ax1.text(i, v+1.5, f'{v:.1f}%', ha='center', fontweight='bold', fontsize=13)
ax1.grid(axis='y', alpha=0.3)

times = [results[n]['time'] for n in names]
ax2.bar(names, times, color=colors, width=0.5)
ax2.set_title('속도 (초)', fontweight='bold', fontsize=13)
for i, v in enumerate(times):
    ax2.text(i, v+0.1, f'{v:.1f}s', ha='center', fontweight='bold', fontsize=13)
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('한글 손글씨 OCR 비교 — 실제 필기체', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════
# 4-2. 줄별 상세 비교 표
# ══════════════════════════════════════════════

easy_lines = [l for l in easy_best['text'].split('\n') if l.strip()]
tess_lines = [l for l in tess_best['text'].split('\n') if l.strip()]
max_n = max(len(GT_LINES), len(easy_lines), len(tess_lines))

print("═" * 80)
print("  📋 줄별 상세 비교")
print("═" * 80)
print(f"  {'#':<3} {'정답':<22} {'EasyOCR':<22} {'Tesseract':<22}")
print(f"  {'─'*69}")

easy_line_sims, tess_line_sims = [], []
for i in range(max_n):
    gt = GT_LINES[i] if i < len(GT_LINES) else ''
    el = easy_lines[i] if i < len(easy_lines) else ''
    tl = tess_lines[i] if i < len(tess_lines) else ''
    es = calc_sim(gt, el) if gt else 0
    ts = calc_sim(gt, tl) if gt else 0
    easy_line_sims.append(es)
    tess_line_sims.append(ts)
    e_mark = '✅' if es > 0.7 else '❌'
    t_mark = '✅' if ts > 0.7 else '❌'
    print(f"  {i+1:<3} {gt:<22} {el:<20}{e_mark} {tl:<20}{t_mark}")

print(f"\n  📊 줄별 평균 정확도:")
print(f"     EasyOCR:  {np.mean(easy_line_sims):.1%}")
print(f"     Tesseract: {np.mean(tess_line_sims):.1%}")

---
# Part 5. CLOVA OCR — 한글 손글씨 최강자

CLOVA OCR은 네이버가 개발한 AI OCR로, **한국어 필기체 인식을 공식 지원**합니다.  
ICDAR 2019 4개 분야 1위, 곡선·기울어진 글자·손글씨까지 인식합니다.

### API 키 발급 절차
1. https://console.ncloud.com 접속 → 로그인
2. `Services` → `AI Services` → `CLOVA OCR` → 이용 신청
3. `Domain` → 도메인 생성 (서비스 타입: **General**)
4. 동작 메뉴 → **"Text OCR"** → API Gateway **자동 연동**
5. **APIGW Invoke URL** + **Secret Key** 복사
6. Colab 🔑 → `CLOVA_OCR_URL`, `CLOVA_OCR_SECRET` 등록

### ⚠️ URL 형식 주의
```
✅ 올바른: https://XXXXX.apigw.ntruss.com/custom/v1/.../general
❌ 잘못된: http://clovaocr-api-kr.ncloud.com/...  (내부망, Colab 불가)
```

### 💰 요금
- General OCR: **월 100건 무료** (초과 건당 3원)
- ⚠️ Template/Document는 도메인 유지비 발생 → 사용 후 삭제!

In [ ]:
# ══════════════════════════════════════════════
# 5-1. CLOVA OCR API 키 로드 + URL 검증
# ══════════════════════════════════════════════
import requests, json, uuid

try:
    from google.colab import userdata
    CLOVA_URL = userdata.get("CLOVA_OCR_URL")
    CLOVA_SECRET = userdata.get("CLOVA_OCR_SECRET")
    print("✅ Colab Secrets에서 로드!")
except:
    CLOVA_URL, CLOVA_SECRET = "", ""

if not CLOVA_URL or not CLOVA_SECRET:
    print("⚠️ API 키가 없습니다.")
    print("   Colab 🔑 → CLOVA_OCR_URL, CLOVA_OCR_SECRET 등록")
    CLOVA_URL = input("   Invoke URL (없으면 Enter): ").strip()
    CLOVA_SECRET = input("   Secret Key (없으면 Enter): ").strip()

# URL 검증
CLOVA_READY = False
if CLOVA_URL:
    from urllib.parse import urlparse
    p = urlparse(CLOVA_URL)
    print(f"\n📋 URL 검증:")
    print(f"   호스트: {p.hostname}")
    print(f"   HTTPS: {'✅' if p.scheme=='https' else '❌ → https 필요!'}")
    print(f"   apigw.ntruss.com: {'✅' if 'apigw.ntruss.com' in (p.hostname or '') else '❌'}")
    if 'apigw.ntruss.com' in (p.hostname or '') and p.scheme == 'https':
        CLOVA_READY = True
        print(f"   → ✅ 사용 가능!")
    else:
        print(f"   → ❌ URL 형식 오류!")
        print(f"     올바른 형식: https://XXX.apigw.ntruss.com/custom/v1/.../general")
else:
    print("\n⚠️ CLOVA OCR 건너뛰기 (Part 5의 나머지 셀 실행 불필요)")

In [ ]:
# ══════════════════════════════════════════════
# 5-2. CLOVA OCR API 호출 + 결과 파싱
# ══════════════════════════════════════════════

def clova_ocr(image_path):
    """CLOVA OCR General API V2 호출"""
    api_url = CLOVA_URL
    if not api_url.startswith('https'):
        api_url = api_url.replace('http://', 'https://')
    if not api_url.endswith('/general'):
        api_url = api_url.rstrip('/') + '/general'

    message = {
        "images": [{"format": image_path.split('.')[-1], "name": "test"}],
        "requestId": str(uuid.uuid4()),
        "version": "V2",
        "timestamp": int(time.time() * 1000),
    }

    with open(image_path, 'rb') as f:
        start = time.time()
        try:
            resp = requests.post(
                api_url,
                headers={'X-OCR-SECRET': CLOVA_SECRET},
                data={'message': json.dumps(message)},
                files=[('file', (image_path, f, f'image/{image_path.split(".")[-1]}'))],
                timeout=30,
            )
            elapsed = time.time() - start
        except requests.exceptions.Timeout:
            print("  ⏰ 타임아웃 (30초)")
            return None, 30
        except Exception as e:
            print(f"  ❌ {e}")
            return None, 0

    if resp.status_code == 200:
        return resp.json(), elapsed
    else:
        print(f"  ❌ {resp.status_code}: {resp.text[:150]}")
        return None, elapsed

def parse_clova(result):
    """CLOVA 응답 → 필드 리스트"""
    fields = []
    if result and 'images' in result:
        for img in result['images']:
            for f in img.get('fields', []):
                fields.append({
                    'text': f.get('inferText', ''),
                    'conf': f.get('inferConfidence', 0),
                    'bbox': f.get('boundingPoly', {}).get('vertices', []),
                    'lineBreak': f.get('lineBreak', False),
                })
    return fields

def fields_to_text(fields):
    """필드 → 전체 텍스트"""
    parts = []
    for f in fields:
        parts.append(f['text'])
        parts.append('\n' if f['lineBreak'] else ' ')
    return ''.join(parts).strip()

if not CLOVA_READY:
    print("⚠️ CLOVA API 키가 없어 건너뜁니다.")
else:
    # 실제 호출
    print("═" * 55)
    print("  🇰🇷 CLOVA OCR 손글씨 인식")
    print("═" * 55)

    result_raw, clova_time = clova_ocr(IMG_PATH)
    clova_fields = parse_clova(result_raw)
    clova_text = fields_to_text(clova_fields)
    clova_sim = calc_sim(GROUND_TRUTH, clova_text)

    print(f"\n  ⏱️ 응답 시간: {clova_time:.2f}초")
    print(f"  📊 전체 정확도: {clova_sim:.1%}")
    print(f"  📝 인식 필드: {len(clova_fields)}개")
    print(f"  📊 평균 신뢰도: {np.mean([f['conf'] for f in clova_fields]):.3f}")
    print(f"\n  📋 인식 결과:")
    for f in clova_fields:
        emoji = '🟢' if f['conf'] > 0.9 else '🟡' if f['conf'] > 0.7 else '🔴'
        lb = ' ↵' if f['lineBreak'] else ''
        print(f"     {emoji} [{f['conf']:.3f}] {f['text']}{lb}")

In [ ]:
# ══════════════════════════════════════════════
# 5-3. CLOVA 바운딩 박스 시각화
# ══════════════════════════════════════════════

if not CLOVA_READY:
    print("⚠️ CLOVA API 미사용 — 건너뜁니다.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # (1) 바운딩 박스 오버레이
    img_vis = Image.open(IMG_PATH).copy()
    draw = ImageDraw.Draw(img_vis)
    sf = ImageFont.truetype(FONT, 11)

    for f in clova_fields:
        bbox = f['bbox']
        if bbox and len(bbox) >= 4:
            pts = [(v.get('x',0), v.get('y',0)) for v in bbox]
            color = '#10B981' if f['conf'] > 0.9 else '#F97316' if f['conf'] > 0.7 else '#E94560'
            draw.polygon(pts, outline=color)
            draw.polygon([(p[0]+1,p[1]+1) for p in pts], outline=color)
            try:
                draw.text((pts[0][0], max(0, pts[0][1]-14)),
                         f'{f["text"][:8]} ({f["conf"]:.0%})', fill=color, font=sf)
            except: pass

    axes[0].imshow(np.array(img_vis))
    axes[0].set_title(f'CLOVA OCR 바운딩 박스\n정확도: {clova_sim:.1%}',
                      fontweight='bold', fontsize=13)
    axes[0].axis('off')

    # (2) 필드별 신뢰도 막대
    field_texts = [f['text'][:10] for f in clova_fields]
    field_confs = [f['conf'] for f in clova_fields]
    bar_colors = ['#10B981' if c>0.9 else '#F97316' if c>0.7 else '#E94560' for c in field_confs]

    y_pos = range(len(field_texts))
    axes[1].barh(y_pos, field_confs, color=bar_colors)
    axes[1].set_yticks(y_pos)
    axes[1].set_yticklabels(field_texts, fontsize=9)
    axes[1].set_xlim(0, 1.05)
    axes[1].axvline(x=0.9, color='gray', linestyle='--', alpha=0.5, label='0.9 기준')
    axes[1].set_title('필드별 신뢰도', fontweight='bold', fontsize=13)
    axes[1].set_xlabel('신뢰도')
    axes[1].legend()
    for i, v in enumerate(field_confs):
        axes[1].text(v+0.01, i, f'{v:.3f}', va='center', fontsize=9)
    axes[1].invert_yaxis()

    plt.suptitle('CLOVA OCR 상세 분석', fontweight='bold', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# ══════════════════════════════════════════════
# 5-4. CLOVA vs EasyOCR vs Tesseract 종합 비교
# ══════════════════════════════════════════════

if not CLOVA_READY:
    print("⚠️ CLOVA API 미사용 — 건너뜁니다.")
else:
    # results 딕셔너리에 CLOVA 추가
    results['CLOVA'] = {'text': clova_text, 'sim': clova_sim, 'time': clova_time}

    # ── 줄별 3종 비교 ──
    clova_lines = [l for l in clova_text.split('\n') if l.strip()]
    easy_lines = [l for l in easy_best['text'].split('\n') if l.strip()]
    tess_lines = [l for l in tess_best['text'].split('\n') if l.strip()]
    max_n = max(len(GT_LINES), len(clova_lines), len(easy_lines), len(tess_lines))

    print("═" * 90)
    print("  📋 줄별 3종 OCR 비교")
    print("═" * 90)
    print(f"  {'#':<3} {'정답':<20} {'CLOVA':<20} {'EasyOCR':<20} {'Tesseract':<20}")
    print(f"  {'─'*83}")

    clova_line_sims, easy_line_sims2, tess_line_sims2 = [], [], []
    for i in range(max_n):
        gt = GT_LINES[i] if i < len(GT_LINES) else ''
        cl = clova_lines[i] if i < len(clova_lines) else ''
        el = easy_lines[i] if i < len(easy_lines) else ''
        tl = tess_lines[i] if i < len(tess_lines) else ''
        cs = calc_sim(gt, cl) if gt else 0
        es = calc_sim(gt, el) if gt else 0
        ts = calc_sim(gt, tl) if gt else 0
        clova_line_sims.append(cs)
        easy_line_sims2.append(es)
        tess_line_sims2.append(ts)
        c_m = '✅' if cs > 0.7 else '❌'
        e_m = '✅' if es > 0.7 else '❌'
        t_m = '✅' if ts > 0.7 else '❌'
        print(f"  {i+1:<3} {gt:<20} {cl:<18}{c_m} {el:<18}{e_m} {tl:<18}{t_m}")

    # ── 종합 수치 ──
    print(f"\n  {'─'*50}")
    print(f"  {'엔진':<12} {'전체 정확도':<12} {'줄별 평균':<12} {'속도'}")
    print(f"  {'─'*50}")
    for name in ['CLOVA', 'EasyOCR', 'Tesseract']:
        r = results[name]
        if name == 'CLOVA': line_avg = np.mean(clova_line_sims)
        elif name == 'EasyOCR': line_avg = np.mean(easy_line_sims2)
        else: line_avg = np.mean(tess_line_sims2)
        emoji = '🟢' if r['sim']>0.7 else '🟡' if r['sim']>0.4 else '🔴'
        print(f"  {emoji} {name:<10} {r['sim']:<12.1%} {line_avg:<12.1%} {r['time']:.2f}초")

In [ ]:
# ══════════════════════════════════════════════
# 5-5. 3종 비교 시각화
# ══════════════════════════════════════════════

if not CLOVA_READY:
    print("⚠️ CLOVA API 미사용 — 건너뜁니다.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    names3 = ['CLOVA', 'EasyOCR', 'Tesseract']
    colors3 = ['#E94560', '#3B82F6', '#10B981']

    # (1) 정확도
    sims3 = [results[n]['sim']*100 for n in names3]
    axes[0].bar(names3, sims3, color=colors3, width=0.5)
    axes[0].set_ylim(0, 105)
    axes[0].set_title('정확도 (%)', fontweight='bold', fontsize=13)
    for i, v in enumerate(sims3):
        axes[0].text(i, v+1.5, f'{v:.1f}%', ha='center', fontweight='bold', fontsize=12)
    axes[0].grid(axis='y', alpha=0.3)

    # (2) 속도
    times3 = [results[n]['time'] for n in names3]
    axes[1].bar(names3, times3, color=colors3, width=0.5)
    axes[1].set_title('속도 (초)', fontweight='bold', fontsize=13)
    for i, v in enumerate(times3):
        axes[1].text(i, v+0.1, f'{v:.1f}s', ha='center', fontweight='bold', fontsize=12)
    axes[1].grid(axis='y', alpha=0.3)

    # (3) 줄별 정확도 히트맵
    line_data = []
    for i in range(len(GT_LINES)):
        cl = clova_lines[i] if i < len(clova_lines) else ''
        el = easy_lines[i] if i < len(easy_lines) else ''
        tl = tess_lines[i] if i < len(tess_lines) else ''
        gt = GT_LINES[i]
        line_data.append([calc_sim(gt,cl), calc_sim(gt,el), calc_sim(gt,tl)])

    line_arr = np.array(line_data)
    im = axes[2].imshow(line_arr, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
    axes[2].set_xticks(range(3))
    axes[2].set_xticklabels(names3, fontsize=10)
    axes[2].set_yticks(range(len(GT_LINES)))
    axes[2].set_yticklabels([f'{i+1}. {l[:8]}...' for i, l in enumerate(GT_LINES)], fontsize=9)
    axes[2].set_title('줄별 정확도 히트맵', fontweight='bold', fontsize=13)
    for i in range(line_arr.shape[0]):
        for j in range(line_arr.shape[1]):
            axes[2].text(j, i, f'{line_arr[i,j]:.0%}', ha='center', va='center',
                        fontsize=10, fontweight='bold',
                        color='white' if line_arr[i,j] < 0.5 else 'black')
    plt.colorbar(im, ax=axes[2], shrink=0.8)

    plt.suptitle('CLOVA vs EasyOCR vs Tesseract — 실제 손글씨', fontweight='bold', fontsize=14)
    plt.tight_layout()
    plt.show()

    # 최종 승자
    best = max(results.items(), key=lambda x: x[1]['sim'])
    print(f"\n  🏆 최고 성능: {best[0]} ({best[1]['sim']:.1%})")

---
## 📝 전체 정리

### 핵심 코드

```python
# EasyOCR (오픈소스, 무료)
reader = easyocr.Reader(['ko', 'en'])
result = reader.readtext('handwriting.png')

# Tesseract (오픈소스, 무료)
text = pytesseract.image_to_string(img, lang='kor', config='--psm 6')

# CLOVA OCR (API, 월 100건 무료)
resp = requests.post(
    'https://XXX.apigw.ntruss.com/.../general',
    headers={'X-OCR-SECRET': SECRET},
    data={'message': json.dumps(msg)},
    files=[('file', open('img.png','rb'))]
)
for field in resp.json()['images'][0]['fields']:
    print(field['inferText'], field['inferConfidence'])
```

### OCR 비교

| 항목 | CLOVA | EasyOCR | Tesseract |
|------|-------|---------|----------|
| 한글 필기체 | **공식 지원** | △ | ✗ |
| 곡선/기울기 | **✅** | △ | ✗ |
| 정확도 | **최고** | 양호 | 보통 |
| 바운딩 박스 | ✅ (4점 좌표) | ✅ (4점 좌표) | △ (단어 박스) |
| 비용 | 100건 무료 | **완전 무료** | **완전 무료** |
| 로컬 실행 | ✗ (API) | ✅ | ✅ |

### CLOVA API 키 주의

| 항목 | 내용 |
|------|------|
| URL 형식 | `https://XXX.apigw.ntruss.com/custom/v1/.../general` |
| Colab 등록 | 🔑 → `CLOVA_OCR_URL`, `CLOVA_OCR_SECRET` |
| General 무료 | 월 100건 (초과 건당 3원) |
| Template/Document | ⚠️ **도메인 유지비** → 사용 후 삭제! |

### 성능 향상 팁
- **전처리**: 대비 향상 + 이진화 (Tesseract에 특히 효과적)
- **EasyOCR**: `gpu=True`로 속도 향상, 원본이 전처리보다 나을 수 있음
- **Tesseract**: `--psm 6` (균일 블록) 또는 `--psm 4` (가변 블록)
- **CLOVA**: 전처리 불필요 (내부 처리 우수)